## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Burger patty weights: mean and range chart

**Xbar-R Control Chart** · a worked example from the StatNex teaching guide

**The story.** A burger chain's kitchen weighs five patties a day, at 09:00, 11:00, 13:00, 15:00 and 17:00, for 25 working days. Patties should weigh about 113 g. On day 16 the scale at the patty press was set wrongly.

**The question.** Is the patty-making process stable, and on which days did something unusual happen?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/xbar_r_chart.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Date,Group number,09:00,11:00,13:00,15:00,17:00
2025-07-01,1.0,112.6,113.0,113.3,112.9,111.8
2025-07-02,2.0,112.8,112.5,113.8,116.1,114.3
2025-07-03,3.0,113.9,114.2,114.2,111.4,113.4
2025-07-04,4.0,112.3,111.8,113.0,114.2,114.7
2025-07-07,5.0,111.5,113.0,113.1,114.5,112.4
2025-07-08,6.0,112.1,111.2,113.7,115.3,113.7
2025-07-09,7.0,113.0,111.1,111.8,110.1,113.1
2025-07-10,8.0,114.4,111.6,113.8,111.5,112.5
2025-07-11,9.0,114.1,113.5,113.5,114.3,113.9
2025-07-14,10.0,113.7,113.5,113.8,113.6,114.9
2025-07-15,11.0,112.2,114.9,112.2,114.1,112.0
2025-07-16,12.0,112.4,113.9,112.4,115.0,112.1
2025-07-17,13.0,115.0,114.1,111.6,113.7,111.1
2025-07-18,14.0,111.1,111.8,116.2,113.2,114.5
2025-07-21,15.0,112.2,110.7,112.8,113.5,113.8
2025-07-22,16.0,115.4,115.2,114.8,119.1,114.7
2025-07-23,17.0,113.6,112.9,113.5,112.0,113.7
2025-07-24,18.0,112.7,112.1,111.3,116.5,111.6
2025-07-25,19.0,113.5,113.1,112.4,113.0,113.4
2025-07-28,20.0,116.0,111.8,110.6,112.8,111.7
2025-07-29,21.0,111.3,112.6,110.9,113.2,110.8
2025-07-30,22.0,112.9,112.0,113.0,113.0,112.5
2025-07-31,23.0,109.8,115.9,111.0,113.0,110.3
2025-08-01,24.0,114.0,110.8,112.3,113.7,112.0
2025-08-04,25.0,113.2,114.3,114.0,112.7,113.6
""")

## Helper: control_signals

A step that no library does, written out. In the code download it is the file `control_signals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
# draw and pattern_signals: defined in the helper cell control_signals above (control_signals.py in the code download)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One row per subgroup (a small sample taken at one time), one column per measurement in
# it (2 to 25 columns). A blank or text cell would change the subgroup's size, so it
# stops the chart instead of being skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["09:00", "11:00", "13:00", "15:00", "17:00"]
values = data[columns].apply(pd.to_numeric, errors="coerce")
if values.isna().any().any():
    raise ValueError("Every cell must hold a number: complete or remove the subgroup.")
values.index = np.arange(1, len(values) + 1)  # subgroup numbers 1, 2, ...
size = len(columns)

## Control-chart constants (JIS Z 9021)

In [ ]:
# Both charts put their limits 3 standard deviations either side of the centre line. The
# process standard deviation is estimated from the mean range R-bar (it is R-bar / d2),
# and these constants turn R-bar into the limits for subgroups of n readings:
#   Xbar chart: CL +/- A2 x R-bar, where A2 = 3 / (d2 sqrt(n))
#   R chart:    UCL = D4 x R-bar and LCL = D3 x R-bar, where D4, D3 = 1 +/- 3 d3 / d2
# (d2 and d3 are the mean and the standard deviation of the range of n normal readings,
# in units of sigma; D3 is 0 where the formula would go below 0). They are the printed
# table's three-decimal values (JIS Z 9021, ASTM, Montgomery's appendix), as StatNex
# uses them, for n = 2..25.
A2 = {2: 1.880, 3: 1.023, 4: 0.729, 5: 0.577, 6: 0.483, 7: 0.419, 8: 0.373, 9: 0.337,
      10: 0.308, 11: 0.285, 12: 0.266, 13: 0.249, 14: 0.235, 15: 0.223, 16: 0.212,
      17: 0.203, 18: 0.194, 19: 0.187, 20: 0.180, 21: 0.173, 22: 0.167, 23: 0.162,
      24: 0.157, 25: 0.153}[size]
D3 = {2: 0.000, 3: 0.000, 4: 0.000, 5: 0.000, 6: 0.000, 7: 0.076, 8: 0.136, 9: 0.184,
      10: 0.223, 11: 0.256, 12: 0.283, 13: 0.307, 14: 0.328, 15: 0.347, 16: 0.363,
      17: 0.378, 18: 0.391, 19: 0.404, 20: 0.415, 21: 0.425, 22: 0.435, 23: 0.443,
      24: 0.452, 25: 0.459}[size]
D4 = {2: 3.267, 3: 2.575, 4: 2.282, 5: 2.114, 6: 2.004, 7: 1.924, 8: 1.864, 9: 1.816,
      10: 1.777, 11: 1.744, 12: 1.717, 13: 1.693, 14: 1.672, 15: 1.653, 16: 1.637,
      17: 1.622, 18: 1.609, 19: 1.596, 20: 1.585, 21: 1.575, 22: 1.565, 23: 1.557,
      24: 1.548, 25: 1.541}[size]

## Points: subgroup mean and range

In [ ]:
# The mean of each subgroup, and its range (largest reading minus smallest).
mean = values.mean(axis=1)
spread = values.max(axis=1) - values.min(axis=1)


def limits(keep):
    """Centre line, upper and lower limits of each chart, from the subgroups in keep."""
    x_bar, r_bar = mean[keep].mean(), spread[keep].mean()
    return {"mean": (x_bar, x_bar + A2 * r_bar, x_bar - A2 * r_bar),
            "range": (r_bar, D4 * r_bar, D3 * r_bar)}


keep = pd.Series(True, index=values.index)

## Control chart data

In [ ]:
# One row per subgroup: its mean against the Xbar chart's centre line (CL) and limits
# (UCL, LCL), then its range against the R chart's. A point outside its limits is a
# signal that something other than everyday variation affected that subgroup. Read the
# R chart first: the Xbar limits are built from R-bar, so they only mean something while
# the spread is stable. n <= 6 makes D3 zero: the R chart then has no lower limit.
lines = limits(keep)
table = pd.DataFrame({"mean": mean[keep]})
table["CL"], table["UCL"], table["LCL"] = lines["mean"]
table["range"] = spread[keep]
table["R CL"], table["R UCL"], table["R LCL"] = lines["range"]
print()
print(table)
outside_now = ((table["mean"] > table["UCL"]) | (table["mean"] < table["LCL"])
               | (table["range"] > table["R UCL"]) | (table["range"] < table["R LCL"]))
print(f"\nSubgroups outside the limits: {list(table.index[outside_now])}")

## The charts

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row. The limits assume a stable process whose subgroup means are roughly
# normal; then a point outside happens by chance about 3 times in 1,000.
for message in (pattern_signals("Xbar chart", table["mean"], table["CL"])
                + pattern_signals("R chart", table["range"], table["R CL"])):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/xbar_chart.png", "Xbar chart", table["mean"], table["CL"], table["UCL"], table["LCL"])
draw("figures/r_chart.png", "R chart", table["range"], table["R CL"], table["R UCL"], table["R LCL"])

## What the result says

This chart follows two things for each day's five patties: the mean weight (Xbar) and the range, which is the gap between the heaviest and the lightest patty. For the mean, the centre line (CL) is 113.08 g, and the control limits, the reach of ordinary chance variation, run from 111.3 g (LCL) to 114.8 g (UCL).

Day 16 has a mean of 115.84 g, above the UCL. No other day is outside. On the same day the range is 4.4 g, well under the range chart's UCL of 6.43 g.

So on day 16 all patties were heavy, but they were no more uneven than usual. That pattern points to a setting, such as the scale, not to careless work. On the other 24 days the process ran steadily.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.